In [26]:
import cv2
import datetime
import csv

# load cascade
face_cascade=cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

# load trained model
recognizer=cv2.face.LBPHFaceRecognizer_create()
recognizer.read("training.yml")


# Label map
label_map={
    0:"Kevin",
    1:'Martin'
}

marked=set()

def marked_attendence(name):
    with open("attendence.csv","a",newline="") as f:
        writer=csv.writer(f)
        time_now=datetime.datetime.now().strftime("%H:%M:%S")
        writer.writerow([name,time_now])

cap=cv2.VideoCapture(0)



while True:
    ret,frame=cap.read()
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    faces=face_cascade.detectMultiScale(gray,scaleFactor=1.3,minNeighbors=5)
    for (x,y,w,h) in faces:
        face_roi=gray[y:y+h,x:x+w]
        face_roi=cv2.resize(face_roi,(200,200))
        label,confidence=recognizer.predict(face_roi)
        if confidence<100:
            name=label_map[label]
            if name not in marked:
                marked_attendence(name)
                marked.add(name)
            cv2.putText(frame,name,(x,y-10),cv2.FONT_HERSHEY_SIMPLEX,1,(123,0,255),2)
        else:
            name="Unknown"
            # cv2.putText(frame,name,(x,y-10),cv2.FONT_HERSHEY_SIMPLEX,1,(123,0,255),2)
        
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.imshow("Attendence Marking System By Kevin Manjila",frame)
    if cv2.waitKey(1)& 0xFF==ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
